## My Contribution – Machine Learning Model Development

This notebook contains both code and explanations related to model selection, training, and evaluation.


In [1]:
from pyspark.sql.functions import col, when
import pandas as pd
from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

df = spark.read.csv("/Workspace/Users/isaramadunika4@gmail.com/Dataset/online_retail_II.csv", header=True, inferSchema=True)
df_clean = (
    df.dropna(subset=["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID"])
      .filter((col("Quantity") > 0) & (col("Price") > 0))
      .withColumn("Sales", col("Quantity") * col("Price"))
      .withColumnRenamed("Invoice", "InvoiceNo")
      .withColumnRenamed("Price", "UnitPrice")
      .withColumnRenamed("Customer ID", "CustomerID")
)

ModuleNotFoundError: No module named 'pyspark'

# Feature Engineering (VectorAssembler)

In [0]:
assembler = VectorAssembler(inputCols=["Quantity", "UnitPrice"], outputCol="features")
df_ml = assembler.transform(df_clean)

# Train-Test Split

In [0]:
train, test = df_ml.randomSplit([0.7, 0.3], seed=42)

Row count: 1067371
Column count: 8
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


# Label Creation

In [ ]:
train_labeled = train.withColumn("label", when(col("Sales") > 50, 1).otherwise(0))
test_labeled = test.withColumn("label", when(col("Sales") > 50, 1).otherwise(0))

# Model Selection
- Decision Tree
- Random Forest
- Logistic Regression

In [0]:
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features")
rf = RandomForestClassifier(labelCol="label", featuresCol="features")
lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=10)

# Model Training

In [0]:
dt_model = dt.fit(train_labeled)
rf_model = rf.fit(train_labeled)
lr_model = lr.fit(train_labeled)

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom,83.4
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,81.0
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,81.0
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom,100.80000000000001
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,30.0
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom,39.599999999999994
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,30.0
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom,59.5
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom,30.599999999999998
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom,45.0


# Predictions

In [0]:
dt_preds = dt_model.transform(test_labeled)
rf_preds = rf_model.transform(test_labeled)
lr_preds = lr_model.transform(test_labeled)

# Evaluation (Accuracy, Precision, Recall)

In [ ]:
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")

results_classification = [
    ["Decision Tree", evaluator_acc.evaluate(dt_preds), evaluator_prec.evaluate(dt_preds), evaluator_rec.evaluate(dt_preds)],
    ["Random Forest", evaluator_acc.evaluate(rf_preds), evaluator_prec.evaluate(rf_preds), evaluator_rec.evaluate(rf_preds)],
    ["Logistic Regression", evaluator_acc.evaluate(lr_preds), evaluator_prec.evaluate(lr_preds), evaluator_rec.evaluate(lr_preds)]
]

# Results Comparison Table

In [ ]:
comparison_df = pd.DataFrame(results_classification, columns=["Model", "Accuracy", "Precision", "Recall"])
display(comparison_df)

# Conclusion (best model)

In [0]:
best_row = max(results_classification, key=lambda row: row[1])
print(f"Best model: {best_row[0]} with accuracy {best_row[1]:.4f}")